# 01. Data Preparation

Converts the raw BLM 2023 Post-Event Inspection coordinates (extracted from the
PDF report as UTM Northing/Easting) into lat/lon, and produces the final
33-point table used by every downstream notebook.

**Input:** `blm_pei_2023_georeferenced_points_v2.csv` (site_id, category,
northing_utm, easting_utm, debris_ft2_per_acre, value_source, notes)

**Output:** `blm_pei_2023_latlon_final.csv` (site_id, category, lat, lon,
debris_ft2_per_acre, value_source, notes)

**Known data issue, already corrected below:** the source PDF's
`northing_utm` and `easting_utm` columns are swapped (confirmed via sanity
check against The_Man's known public location). The conversion function
below swaps them back before projecting to lat/lon — do not "fix" this by
removing the swap, it is intentional.


## Mount Drive and upload the raw extracted points

In [1]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [2]:
from google.colab import files
uploaded = files.upload()  # select blm_pei_2023_georeferenced_points_v2.csv


Saving blm_pei_2023_georeferenced_points_v2.csv to blm_pei_2023_georeferenced_points_v2.csv


In [3]:
!pip install pyproj -q

import pandas as pd

df = pd.read_csv("blm_pei_2023_georeferenced_points_v2.csv")
df.head()


,site_id,category,northing_utm,easting_utm,debris_ft2_per_acre,value_source,notes
0,The_Man,POI,314078,4517380,0.10,Appendix B (p.8),POI since 2013; not included in 2023 official ...
1,The_Temple,POI,314632,4517906,0.76,Appendix B (p.8),POI since 2013; not included in 2023 official ...
2,USS,POI,312339,4516381,0.90,Appendix B (p.8),POI since 2013; not included in 2023 official ...
3,DPW,POI,313039,4515697,0.02,Appendix B (p.8),POI since 2013; not included in 2023 official ...
4,JOC,POI,309978,4515553,0.29,Appendix B (p.8),POI since 2013; not included in 2023 official ...


## Convert UTM coordinates to lat/lon (with the swapped-column fix)

In [4]:
from pyproj import Transformer

# NAD83 / UTM Zone 11N
transformer_nad83 = Transformer.from_crs("EPSG:26911", "EPSG:4326", always_xy=True)

# WGS84 / UTM Zone 11N
transformer_wgs84 = Transformer.from_crs("EPSG:32611", "EPSG:4326", always_xy=True)

def convert_row(row, transformer):
    # NOTE: report's "northing_utm" column is actually the easting value,
    # and "easting_utm" column is actually the northing value (columns swapped in source PDF)
    true_easting = row["northing_utm"]
    true_northing = row["easting_utm"]
    lon, lat = transformer.transform(true_easting, true_northing)
    return pd.Series({"lat": lat, "lon": lon})

df[["lat_nad83", "lon_nad83"]] = df.apply(
    lambda r: convert_row(r, transformer_nad83), axis=1
).rename(columns={"lat": "lat_nad83", "lon": "lon_nad83"})

df[["lat_wgs84", "lon_wgs84"]] = df.apply(
    lambda r: convert_row(r, transformer_wgs84), axis=1
).rename(columns={"lat": "lat_wgs84", "lon": "lon_wgs84"})

# Sanity check: The_Man should land near 40.786, -119.206.
# NAD83 and WGS84 give effectively identical results at this location, so
# either transformer's output is fine to use going forward.
print(df[df["site_id"] == "The_Man"][["lat_nad83", "lon_nad83", "lat_wgs84", "lon_wgs84"]])


   lat_nad83   lon_nad83  lat_wgs84   lon_wgs84
0  40.786383 -119.203505  40.786383 -119.203505


## Finalize and save the 33-point table used by all later notebooks

In [5]:
df["lat"] = df["lat_nad83"]
df["lon"] = df["lon_nad83"]

df_final = df[["site_id", "category", "lat", "lon", "debris_ft2_per_acre", "value_source", "notes"]]

import os
DATA_DIR = "/content/drive/MyDrive/burning_man_repo/data"
os.makedirs(DATA_DIR, exist_ok=True)
df_final.to_csv(os.path.join(DATA_DIR, "blm_pei_2023_latlon_final.csv"), index=False)
print(f"Saved to {DATA_DIR}/blm_pei_2023_latlon_final.csv")

df_final.head()


Saved to /content/drive/MyDrive/burning_man_repo/data/blm_pei_2023_latlon_final.csv


,site_id,category,lat,lon,debris_ft2_per_acre,value_source,notes
0,The_Man,POI,40.786383,-119.203505,0.10,Appendix B (p.8),POI since 2013; not included in 2023 official ...
1,The_Temple,POI,40.791243,-119.197100,0.76,Appendix B (p.8),POI since 2013; not included in 2023 official ...
2,USS,POI,40.776995,-119.223801,0.90,Appendix B (p.8),POI since 2013; not included in 2023 official ...
3,DPW,POI,40.770997,-119.215307,0.02,Appendix B (p.8),POI since 2013; not included in 2023 official ...
4,JOC,POI,40.768999,-119.251507,0.29,Appendix B (p.8),POI since 2013; not included in 2023 official ...
